# Diabetic retinopathy model: starter code

The model has two stages:

1. A **vision model** (ResNet50, `model.h5`) grades the retina image and outputs a probability for each of the five grades.
2. A **random forest** (`models/stacker_rf.joblib`) combines these probabilities with **patient data** into the final grade.

`retinopathy_model.py` contains the code to load and run both stages. Download `model.h5` and the `images`
folder from the Google Drive folder linked in the README and adjust the paths below if needed.

In [ ]:
import matplotlib.pyplot as plt

from retinopathy_model import GRADES, RetinopathyModel, load_image, load_patient

CNN_PATH = "model.h5"                       # vision model, from Google Drive
STACKER_PATH = "models/stacker_rf.joblib"   # random forest, in this repository
IMAGE_DIR = "images"                        # retina images, from Google Drive

model = RetinopathyModel(CNN_PATH, STACKER_PATH)

## Patient data

Each patient has an id, which is also the name of their retina image. `load_patient` looks up the
patient data in `data/train.csv` and `data/test.csv`.

In [ ]:
patient = load_patient("281d7b7c7676")
patient

In [ ]:
image = load_image(f"{IMAGE_DIR}/{patient.id_code}.png")
plt.imshow(image)
plt.axis("off")
plt.show()

## Prediction

In [ ]:
prediction = model.predict(image, patient)

print(f"final grade:              {prediction.grade_name}")
print(f"grade from image alone:   {GRADES[prediction.cnn_grade]}")
print(f"grade by clinician:       {GRADES[patient.diagnosis]}")
print()
print(f"{'grade':18s} {'image alone':>12s} {'final':>8s}")
for grade, name in GRADES.items():
    print(f"{name:18s} {prediction.cnn_probabilities[grade]:12.2f} {prediction.probabilities[grade]:8.2f}")

## Explanations

Generate explanations here and fill them into an HTML template.

In [ ]:
explanation = "explanation goes here..."


def generate_html(name, image, patient, prediction, explanation):
    html = open("explanation_template.html").read()
    html = html.replace("PXX", patient.id_code)
    html = html.replace("GXX", "Male" if patient.gender == "M" else "Female")
    html = html.replace("AXX", str(patient.age))

    plt.imshow(image)
    plt.axis("off")
    plt.savefig("IMAGE_FOR_DISPLAY.png")

    html = html.replace("DXX", prediction.grade_name)
    html = html.replace("EXPLANATION", explanation)
    with open(name, "w") as fp:
        fp.write(html)


generate_html("test.html", image, patient, prediction, explanation)